In [1]:
import numpy as np
import scipy.optimize as opt
from scipy.stats import qmc
import xarray as xr
import opt_function as opt_func
from pathlib import Path
module_dir = Path("/glade/work/iranjan/tpose24-osse/")

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
TRUE_W = "/glade/work/iranjan/fast-osse/fastosse-run1/true_w.nc"
PMO_SCRIPT = "/glade/work/iranjan/DART/models/MOM6/work/perfect_model_obs"
true_w = xr.open_dataset(TRUE_W)

In [4]:
N_GLIDERS = 6
lat_bounds = (-1, 2.0)
lon_bounds = (218.5, 221.5)
 
# scipy.optimize.Bounds wants one (min, max) pair per parameter, flattened.
# Since x is [lat0, lon0, lat1, lon1, ...], lower/upper_bounds repeat the
# same lat/lon range once per glider.
lower_bounds = []
upper_bounds = []
for _ in range(N_GLIDERS):
    lower_bounds.extend([lat_bounds[0], lon_bounds[0]])
    upper_bounds.extend([lat_bounds[1], lon_bounds[1]])
bounds = opt.Bounds(lower_bounds, upper_bounds)

sampler = qmc.LatinHypercube(d=2, seed=42)
unit_samples = sampler.random(n=N_GLIDERS)  # (N_GLIDERS, 2), each in [0,1)
lat_range = lat_bounds[1] - lat_bounds[0]
lon_range = lon_bounds[1] - lon_bounds[0]
x0 = np.empty(N_GLIDERS * 2)
x0[0::2] = lat_bounds[0] + unit_samples[:, 0] * lat_range
x0[1::2] = lon_bounds[0] + unit_samples[:, 1] * lon_range

In [5]:
"""def objective_function(x, true_w_dataset, min_dist_deg=0.15):
    
    Wrapper that unpacks glider positions, checks for spatial overlap,
    and executes opt_loc.
    
    loc_matrix = x.reshape(N_GLIDERS, 2)
    loc_list = loc_matrix.tolist()

    # distance penalty between every pair of gliders
    too_close = False
    closest_distance = float('inf')
    for i in range(N_GLIDERS):
        for j in range(i + 1, N_GLIDERS):
            dist = np.sqrt((loc_matrix[i, 0] - loc_matrix[j, 0]) ** 2 +
                            (loc_matrix[i, 1] - loc_matrix[j, 1]) ** 2)
            closest_distance = min(closest_distance, dist)
            if dist < min_dist_deg:
                too_close = True

    if too_close:
        print(f"--> SKIPPING RUN (Gliders too close! Closest pair distance: {closest_distance:.4f}°)")
        return 1e6 + (min_dist_deg - closest_distance) * 1e5

    w_diff = opt_func.opt_loc(loc_list, true_w_dataset)
    norm_error = np.linalg.norm(w_diff.values if hasattr(w_diff, 'values') else w_diff)
    print(f"--> Resulting L2 Norm Error: {norm_error:.6e}")
    return norm_error"""


def objective_function(x, true_w_dataset, min_dist_deg=0.15):
    """
    Wrapper scipy.optimize.minimize actually calls. Unpacks the flat
    parameter vector into glider (lat, lon) pairs, rejects configurations
    where any two gliders are too close (returning a penalty instead of
    running the expensive PMO pipeline for a physically degenerate
    configuration), and otherwise calls opt_loc and reduces its output to
    a scalar loss.
    """
    loc_matrix = x.reshape(N_GLIDERS, 2)
    loc_list = loc_matrix.tolist()
 
    # --- distance penalty between every pair of gliders ---
    """too_close = False
    closest_distance = float('inf')
    for i in range(N_GLIDERS):
        for j in range(i + 1, N_GLIDERS):
            dist = np.sqrt((loc_matrix[i, 0] - loc_matrix[j, 0]) ** 2 +
                            (loc_matrix[i, 1] - loc_matrix[j, 1]) ** 2)
            closest_distance = min(closest_distance, dist)
            if dist < min_dist_deg:
                too_close = True
 
    if too_close:
        print(f"--> SKIPPING RUN (Gliders too close! Closest pair distance: {closest_distance:.4f}°)")
        return 1e6 + (min_dist_deg - closest_distance) * 1e5"""
 
    # opt_loc now handles NaN internally (nan-safe norm, or 1e10 if the
    # diff is entirely NaN) and returns a plain scalar — no further
    # reduction needed here.
    norm_error = opt_func.opt_loc(loc_list, true_w_dataset)
    print(f"--> Resulting L2 Norm Error: {norm_error:.6e}")
    return norm_error

In [6]:
%%time
result1 = objective_function(x0, true_w, min_dist_deg=0.15)

Found 25 unique date groups spanning 2015-01-02 12:00:00 to 2015-06-19 12:00:00.
Found 25 subdirectories, running with MAX_CONCURRENT=25
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-05-003
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-06-014
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-02-008
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-05-010
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-06-021
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-05-031
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-004
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-04-005
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-02-015
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-03-008
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-018
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-025
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-03-022
SUCCESS: EEP_MITgc

In [ ]:
%%time

res = opt.minimize(
    fun=objective_function,
    x0=x0,
    args=(true_w, 0.15),
    bounds=bounds,
    method='Nelder-Mead',
    options={'disp': True, 'maxiter': 800},
)

best_loc_list = res.x.reshape(N_GLIDERS, 2).tolist()
print("\n================ OPTIMIZATION COMPLETE ================")
print(f"Optimization Status: {res.message}")
print(f"Minimum Error Norm Reached: {res.fun}")
print("Best Glider Tracking Coordinates Found:")
for idx, (lat, lon) in enumerate(best_loc_list):
    print(f" Glider {idx+1}: Lat={lat:.6f}, Lon={lon:.6f}")


# 1. Create the 'results' folder if it doesn't already exist
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
output_file = results_dir / "optimization_results.txt"

# 3. Construct the output text block
output_text = []
output_text.append("\n================ OPTIMIZATION COMPLETE ================")
output_text.append(f"Optimization Status: {res.message}")
output_text.append(f"Minimum Error Norm Reached: {res.fun}")
output_text.append("Best Glider Tracking Coordinates Found:")
for idx, (lat, lon) in enumerate(best_loc_list):
    output_text.append(f"  Glider {idx+1}: Lat={lat:.6f}, Lon={lon:.6f}")

# Join lines together into a single block of text
final_output = "\n".join(output_text)

# 4. Print to the console
print(final_output)

# 5. Save to the new text file
output_file.write_text(final_output)


Found 25 unique date groups spanning 2015-01-02 12:00:00 to 2015-06-19 12:00:00.
Found 25 subdirectories, running with MAX_CONCURRENT=25
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-018
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-02-022
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-06-014
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-04-019
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-02-015
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-06-007
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-01-011
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-03-008
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-03-029
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-05-003
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-05-010
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-03-015
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-04-005
SUCCESS: EEP_MITgc

In [ ]:
print(res.nit)

In [ ]:
print(res.nfev)